# A5 — Per-Language and Per-Agent Stratification

**Reviewer concern addressed:** Section A5 of the revision checklist — *"The five agents have very different documentation rates (Codex 14% vs Copilot 72%) and come from different repository bases. Pooling them masks heterogeneity. Add stratified tables or a mixed-effects model with language and agent as random intercepts."*

This notebook:
1. **Per-agent breakdown** — reproduces key RQ1/RQ2 metrics for each of the five agents vs. the developer baseline.
2. **Per-language breakdown** — same metrics grouped by programming language (languages with ≥ 50 functions in *both* cohorts only).
3. **Mixed-effects robustness check** — fits a `statsmodels` linear mixed-effects model with `author_type` as fixed effect and `repo` as random intercept.
4. Saves all summary tables to `revision_outputs/stratified_*.csv`.

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import warnings
warnings.filterwarnings('ignore')

import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, spearmanr
import statsmodels.formula.api as smf

sns.set_theme(style='whitegrid')
PALETTE = {'Agent': '#A7C7E7', 'Developer': '#BDE5B8'}

DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), 'dataset', 'data', 'updated_dataset_metrics.csv')
OUT_DIR   = os.path.join(os.getcwd(), 'revision_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

df_raw = pd.read_csv(DATA_PATH)
print(f"Raw: {df_raw.shape}")

numeric_cols = [
    'doc_lines', 'doc_entropy', 'doc_code_overlap', 'doc_redundancy',
    'cyclomatic_complexity', 'sloc', 'semgrep_findings_count'
]
for col in numeric_cols:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

# ── Tokenizer ─────────────────────────────────────────────────────────────────
def tokenize(text):
    if not isinstance(text, str):
        return []
    return re.findall(r'[A-Za-z_][A-Za-z0-9_]*', text.lower())

# ── Detect programming language from file extension ───────────────────────────
EXT_TO_LANG = {
    '.py': 'Python', '.js': 'JavaScript', '.ts': 'TypeScript',
    '.jsx': 'JavaScript', '.tsx': 'TypeScript',
    '.java': 'Java', '.go': 'Go', '.rb': 'Ruby',
    '.cs': 'C#', '.cpp': 'C++', '.c': 'C',
    '.rs': 'Rust', '.php': 'PHP', '.swift': 'Swift',
    '.kt': 'Kotlin', '.scala': 'Scala', '.sh': 'Shell',
}

def file_lang(path):
    if not isinstance(path, str):
        return 'Unknown'
    ext = os.path.splitext(path)[1].lower()
    return EXT_TO_LANG.get(ext, 'Other')

df_raw['language'] = df_raw['file_path'].apply(file_lang)

# ── Documented-function dataset ───────────────────────────────────────────────
df = df_raw.dropna(subset=['doc_entropy', 'doc_code_overlap', 'doc_redundancy']).copy()
df = df[df['doc_lines'] > 0].copy()
df['doc_tokens'] = df['doc_text'].apply(lambda x: len(tokenize(x)))

print(f"Documented-function dataset: {df.shape}")
print(f"Group counts:")
print(df.groupby(['group', 'label']).size().to_string())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. DOCUMENTATION RATE  (computed before dropping undocumented rows)
# ─────────────────────────────────────────────────────────────────────────────
print('=== Documentation Rate by Agent ===')

doc_rate_rows = []
for lbl, grp in df_raw[df_raw['group'] == 'agent'].groupby('label'):
    total  = len(grp)
    documented = (grp['doc_lines'] > 0).sum()
    doc_rate_rows.append({'agent': lbl, 'total_functions': total,
                          'documented': documented, 'doc_rate': documented / total})

# Developer baseline
hgrp  = df_raw[df_raw['group'] == 'human']
h_doc = (hgrp['doc_lines'] > 0).sum()
doc_rate_rows.append({'agent': 'Developer (baseline)', 'total_functions': len(hgrp),
                      'documented': h_doc, 'doc_rate': h_doc / len(hgrp)})

doc_rate_df = pd.DataFrame(doc_rate_rows).sort_values('doc_rate', ascending=False)
display(doc_rate_df)

doc_rate_df.to_csv(os.path.join(OUT_DIR, 'stratified_doc_rate.csv'), index=False)
print(f"Saved stratified_doc_rate.csv")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. PER-AGENT BREAKDOWN of key RQ1 / RQ2 metrics
# ─────────────────────────────────────────────────────────────────────────────
print('=== Per-Agent Metric Breakdown ===')

metrics = {
    'doc_tokens':          'Median Doc Tokens',
    'doc_code_overlap':    'Median Code Overlap',
    'doc_entropy':         'Median Entropy',
    'doc_redundancy':      'Median Redundancy',
    'cyclomatic_complexity': 'Median CC',
    'sloc':                'Median SLOC',
}

def spearman_rho(series_x, series_y):
    mask = series_x.notna() & series_y.notna()
    if mask.sum() < 5:
        return np.nan
    return spearmanr(series_x[mask], series_y[mask]).statistic

agent_rows = []

for lbl in ['Claude_Code', 'Copilot', 'Cursor', 'Devin', 'OpenAI_Codex']:
    sub = df[df['label'] == lbl]
    row = {'cohort': lbl, 'n': len(sub)}
    for col, name in metrics.items():
        row[name] = sub[col].median()
    row['Spearman rho (tokens~SLOC)'] = spearman_rho(sub['doc_tokens'], sub['sloc'])
    row['Spearman rho (tokens~CC)']   = spearman_rho(sub['doc_tokens'], sub['cyclomatic_complexity'])
    agent_rows.append(row)

# Developer baseline
hu = df[df['group'] == 'human']
row = {'cohort': 'Developer (baseline)', 'n': len(hu)}
for col, name in metrics.items():
    row[name] = hu[col].median()
row['Spearman rho (tokens~SLOC)'] = spearman_rho(hu['doc_tokens'], hu['sloc'])
row['Spearman rho (tokens~CC)']   = spearman_rho(hu['doc_tokens'], hu['cyclomatic_complexity'])
agent_rows.append(row)

agent_breakdown = pd.DataFrame(agent_rows)
display(agent_breakdown.to_string(index=False, float_format='{:.3f}'.format))

agent_breakdown.to_csv(os.path.join(OUT_DIR, 'stratified_per_agent.csv'), index=False)
print('Saved stratified_per_agent.csv')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. PER-LANGUAGE BREAKDOWN (≥50 functions in both cohorts)
# ─────────────────────────────────────────────────────────────────────────────
print('=== Per-Language Metric Breakdown ===')

MIN_N = 50

lang_counts = df.groupby(['language', 'group']).size().unstack(fill_value=0)
print('\nFunctions per language per group (documented only):')
display(lang_counts)

# Qualify: both agent and human must have ≥ MIN_N
if 'agent' in lang_counts.columns and 'human' in lang_counts.columns:
    eligible = lang_counts[(lang_counts['agent'] >= MIN_N) & (lang_counts['human'] >= MIN_N)].index.tolist()
else:
    eligible = []

excluded = [l for l in lang_counts.index if l not in eligible]
print(f"\nIncluded languages ({len(eligible)}): {eligible}")
print(f"Excluded languages ({len(excluded)} — fewer than {MIN_N} functions in at least one cohort): {excluded}")

lang_rows = []
for lang in eligible:
    for grp, grp_name in [('agent', 'Agent'), ('human', 'Developer')]:
        sub = df[(df['language'] == lang) & (df['group'] == grp)]
        row = {'language': lang, 'cohort': grp_name, 'n': len(sub)}
        for col, name in metrics.items():
            row[name] = sub[col].median()
        row['Spearman rho (tokens~SLOC)'] = spearman_rho(sub['doc_tokens'], sub['sloc'])
        row['Spearman rho (tokens~CC)']   = spearman_rho(sub['doc_tokens'], sub['cyclomatic_complexity'])
        lang_rows.append(row)

if lang_rows:
    lang_breakdown = pd.DataFrame(lang_rows)
    display(lang_breakdown.to_string(index=False, float_format='{:.3f}'.format))
    lang_breakdown.to_csv(os.path.join(OUT_DIR, 'stratified_per_language.csv'), index=False)
    print('Saved stratified_per_language.csv')
else:
    print(f'No languages meet the minimum n={MIN_N} threshold in both cohorts. '
          'Consider lowering MIN_N or reporting a single-language table as supplementary.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. MANN-WHITNEY TESTS BY AGENT (each agent vs. developer baseline)
# ─────────────────────────────────────────────────────────────────────────────
print('=== Per-Agent Mann-Whitney Tests (doc_tokens vs developer baseline) ===')

mw_rows = []
hu_tokens = df[df['group'] == 'human']['doc_tokens'].dropna()

for lbl in ['Claude_Code', 'Copilot', 'Cursor', 'Devin', 'OpenAI_Codex']:
    ag_tokens = df[df['label'] == lbl]['doc_tokens'].dropna()
    if len(ag_tokens) < 5:
        continue
    stat, p = mannwhitneyu(ag_tokens, hu_tokens, alternative='two-sided')
    r = 1 - (2 * stat) / (len(ag_tokens) * len(hu_tokens))
    mw_rows.append({
        'agent': lbl,
        'n_agent': len(ag_tokens),
        'n_developer': len(hu_tokens),
        'agent_median_tokens': ag_tokens.median(),
        'developer_median_tokens': hu_tokens.median(),
        'U_stat': stat, 'p_value': p, 'rank_biserial_r': r
    })
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    print(f"  {lbl:15s}  n={len(ag_tokens):4d}  med={ag_tokens.median():6.1f}  r={r:.4f}  p={p:.3e} {sig}")

mw_df = pd.DataFrame(mw_rows)
mw_df.to_csv(os.path.join(OUT_DIR, 'stratified_per_agent_mw.csv'), index=False)
print('Saved stratified_per_agent_mw.csv')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. MIXED-EFFECTS MODEL (robustness check)
#    Outcome: log(doc_tokens + 1)
#    Fixed:   author_type (agent vs developer)
#    Random:  repo (random intercept)
# ─────────────────────────────────────────────────────────────────────────────
print('\n=== Mixed-Effects Model: log(doc_tokens+1) ~ author_type + (1|repo) ===')

df_lme = df[['doc_tokens', 'group', 'repo']].dropna().copy()
df_lme['log_doc_tokens'] = np.log1p(df_lme['doc_tokens'])
df_lme['author_type']    = (df_lme['group'] == 'agent').astype(int)  # 1=agent, 0=human

# Encode repo as numeric group for statsmodels
df_lme['repo_id'] = df_lme['repo'].astype('category').cat.codes

try:
    lme_model  = smf.mixedlm('log_doc_tokens ~ author_type', data=df_lme, groups=df_lme['repo_id'])
    lme_result = lme_model.fit(reml=True, method='lbfgs')
    print(lme_result.summary())

    # Extract the key coefficient
    coef  = lme_result.params['author_type']
    pval  = lme_result.pvalues['author_type']
    ci_lo = lme_result.conf_int().loc['author_type', 0]
    ci_hi = lme_result.conf_int().loc['author_type', 1]

    direction = 'MORE' if coef > 0 else 'FEWER'
    sig_str   = 'SIGNIFICANT' if pval < 0.05 else 'NOT significant'

    print(f"\n--- Key result ---")
    print(f"agent_type coefficient: {coef:.4f}  (95% CI: [{ci_lo:.4f}, {ci_hi:.4f}])")
    print(f"p-value: {pval:.4e}  → {sig_str}")
    print(f"Interpretation: after accounting for repository-level variation, agents produce")
    print(f"{direction} documentation tokens (log-scale) than developers (β = {coef:.3f}).")

    lme_summary = pd.DataFrame([{
        'model': 'log_doc_tokens ~ author_type + (1|repo)',
        'fixed_effect': 'author_type (1=agent)',
        'coefficient': coef, 'ci_lower': ci_lo, 'ci_upper': ci_hi,
        'p_value': pval, 'significant': pval < 0.05
    }])
    lme_summary.to_csv(os.path.join(OUT_DIR, 'stratified_lme_result.csv'), index=False)
    print('Saved stratified_lme_result.csv')

except Exception as e:
    print(f'Mixed-effects model failed: {e}')
    print('Tip: if convergence fails, try method="cg" or reduce groups with fewer observations.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. VISUALISATION — per-agent doc_tokens distribution
# ─────────────────────────────────────────────────────────────────────────────
df_plot = df.copy()
df_plot['cohort'] = df_plot['label'].fillna('Developer')
df_plot.loc[df_plot['group'] == 'human', 'cohort'] = 'Developer'

fig, ax = plt.subplots(figsize=(12, 5), dpi=300)
order = ['Claude_Code', 'Copilot', 'Cursor', 'Devin', 'OpenAI_Codex', 'Developer']
order = [o for o in order if o in df_plot['cohort'].unique()]

sns.boxplot(
    data=df_plot, x='cohort', y='doc_tokens', order=order,
    showfliers=False, ax=ax,
    palette=['#A7C7E7'] * 5 + ['#BDE5B8']
)
ax.set_title('Documentation Token Count by Agent / Cohort', fontsize=16, fontweight='bold')
ax.set_xlabel('', fontsize=14)
ax.set_ylabel('Doc Tokens', fontsize=14)
ax.tick_params(labelsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'stratified_per_agent_boxplot.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved stratified_per_agent_boxplot.png')

## Revision note

- **What this adds:** Stratified tables showing that aggregate agent/developer differences are not solely driven by one agent or one repository base. The mixed-effects model provides a repository-controlled estimate of the documentation-length effect.
- **Where to cite in paper:** Add a subsection or table to RQ1/RQ2 (Section 4.1–4.2). The LME result can be reported as "to control for repository-level clustering, we fitted a linear mixed-effects model (β = X, p = Y), confirming the direction and significance of the aggregate finding."
- **What to watch for:** If a single agent (e.g., OpenAI Codex, n≈53 documented functions) drives a disproportionate result, note this as a limitation and consider sensitivity analyses excluding that agent.